# 02 · Claude Code por comandos: una flag por celda
`claude -p` (no interactivo). Base en todas las celdas: `--model claude-sonnet-5 --setting-sources "" --strict-mcp-config --no-session-persistence`
(sin settings de usuario, sin MCP del usuario). Pregunta fija: *¿Cómo cambió el margen por proyecto de junio a julio, y por qué?*.

Equivalencia con `pi`: `--tools ""` ≈ `-nt` · `--disable-slash-commands` ≈ `-ns` · `--strict-mcp-config` ≈ `-ne` · `--setting-sources ""` ≈ `-nc`. `--bare` = todo a la vez (auth solo por `ANTHROPIC_API_KEY` o `--settings`).

In [1]:
import os, json, logging, warnings
from pathlib import Path

logging.getLogger("anthropic").setLevel(logging.ERROR); warnings.filterwarnings("ignore")

if "ANTHROPIC_API_KEY" not in os.environ:
    raise SystemExit("Falta ANTHROPIC_API_KEY en el entorno.")

MODEL = "claude-sonnet-5"
WS = Path("workspace").resolve()          # contabilidad.csv (sintético) + .claude/skills/
(WS / "CLAUDE.md").unlink(missing_ok=True) # cada corrida empieza sin contexto de proyecto
PREGUNTA = '¿Cómo cambió el margen por proyecto de junio a julio, y por qué?'
SYSTEM = """Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador ';', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto."""
SYSTEM_CORTO = 'Eres un analista financiero. Español, corto.'

import subprocess, shlex

os.environ["CLAUDE_CODE_DISABLE_AUTO_MEMORY"] = "1"   # sin la memoria del usuario: el punto de partida es el modelo solo
BASE = ["--no-session-persistence", "--model", MODEL, "--setting-sources", "", "--strict-mcp-config"]

def cc(prompt=PREGUNTA, *flags, timeout=420):
    """claude -p con aislamiento explícito + las flags del paso."""
    print("$ claude -p '…' " + " ".join(shlex.quote(f) for f in flags) + "\n")
    r = subprocess.run(["claude", "-p", prompt, *BASE, *flags], cwd=WS, capture_output=True, text=True,
                       stdin=subprocess.DEVNULL, timeout=timeout)
    err = [l for l in r.stderr.splitlines() if "takes precedence" not in l]
    print(r.stdout.strip())
    if err:
        print("[stderr]", *err, sep="\n")
    return r.stdout

## 1 · Solo el modelo
Sin tools, sin skills, sin system prompt (`--system-prompt ""` quita el de Claude Code).

In [2]:
cc(PREGUNTA, "--tools", "", "--disable-slash-commands", "--system-prompt", "");

$ claude -p '…' --tools '' --disable-slash-commands --system-prompt ''



No tengo acceso a datos financieros o de proyectos tuyos (no hay ningún archivo, hoja de cálculo o sistema conectado en esta conversación). Para poder comparar el margen por proyecto entre junio y julio y explicar por qué cambió, necesitaría que me compartas la información, por ejemplo:

- Un archivo (Excel/CSV/PDF) con ingresos, costos y margen por proyecto de ambos meses.
- O los datos pegados directamente en el chat (proyecto, ingresos, costos, margen de junio vs. julio).

Si me lo compartes, puedo:
1. Calcular la variación de margen (absoluta y %) por proyecto.
2. Identificar qué proyectos mejoraron o empeoraron más.
3. Sugerir posibles causas (cambios en costos, horas facturadas, tarifas, alcance, etc.) si incluyes ese detalle.

¿Tienes ese archivo o esos datos a mano para compartirlos?


## 2 · + system prompt
`--system-prompt` reemplaza el de Claude Code. `--append-system-prompt` lo extiende.

In [3]:
cc(PREGUNTA, "--tools", "", "--disable-slash-commands", "--system-prompt", SYSTEM);

$ claude -p '…' --tools '' --disable-slash-commands --system-prompt 'Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador '"'"';'"'"', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto.'



Voy a revisar el archivo para calcular los márgenes.

**Herramienta:** Ejecutaré un análisis del CSV con Python/pandas para obtener cifras exactas.

*[Nota: no tengo acceso directo a ejecutar código en este entorno de chat, así que voy a leer el archivo y calcular manualmente]*


## 3 · + Bash
`--tools` = qué existe. `--allowedTools` = qué corre sin preguntar.

In [4]:
cc(PREGUNTA, "--system-prompt", SYSTEM, "--disable-slash-commands",
   "--tools", "Bash", "--allowedTools", "Bash");

$ claude -p '…' --system-prompt 'Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador '"'"';'"'"', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto.' --disable-slash-commands --tools Bash --allowedTools Bash



## Verificación de fuente
- Datos leídos de `./contabilidad.csv` (3060 filas), períodos 6 (junio) y 7 (julio).
- Ingreso = Σ(Credito−Debito) cuentas que empiezan por "4"; Costo = Σ(Debito−Credito) cuentas "6" y "7"; Margen = (Ingreso−Costo)/Ingreso.
- Proyecto "5001" y "0" no tienen ingresos (solo cuentas 5xxx/costos admin) → margen no aplicable, excluidos del análisis.

## Margen por proyecto: junio → julio

| Proyecto | Margen Jun | Margen Jul | Cambio | Causa principal |
|---|---|---|---|---|
| **2210** | 38,0% | 48,0% | ▲ +10,0 pp | Ingreso subió fuerte (+$85M, servicio transporte) más que el costo (mantenimiento vehículos +$25,9M) |
| **1030** | 30,6% | 33,0% | ▲ +2,4 pp | Ingreso (almacenamiento) +$11,3M; costo casi estable (arriendo bodega bajó $9,3M compensando alza en mantenimiento) |
| **3322** | 30,0% | 29,7% | ▼ −0,3 pp | Prácticamente sin cambios (ingreso y costo bajaron proporcionalmente) |
| **4410** | 30,1% | 28,0% | ▼ −2,1 pp | Ingreso subió +$15M pero mantenimiento ve

## 4 · + Read, Glob, Grep

In [5]:
cc(PREGUNTA, "--system-prompt", SYSTEM, "--disable-slash-commands",
   "--tools", "Bash,Read,Glob,Grep", "--allowedTools", "Bash,Read,Glob,Grep");

$ claude -p '…' --system-prompt 'Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador '"'"';'"'"', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto.' --disable-slash-commands --tools Bash,Read,Glob,Grep --allowedTools Bash,Read,Glob,Grep



## Margen por proyecto — Junio vs Julio

**Cálculo:** Margen = (Ingreso[cta 4] − Costo[ctas 6/7]) / Ingreso, por ProjectId y Period.

| Proyecto (CC) | Margen Jun | Margen Jul | Δ | Causa principal |
|---|---|---|---|---|
| **1030** (WAREHOUSE) | 30,6% | 33,0% | ▲ leve | Ingreso similar; menor arriendo bodega (37M→28M) compensa alza en mantenimiento vehículos. |
| **1045** (WAREHOUSE) | 33,7% | 11,5% | ▼▼ fuerte | Ingreso estable (~223M→230M) pero **costo de mercancía vendida se duplicó** (59M→117M) y sueldos subieron (22M→41M). |
| **2210** (FIRST MILE) | 38,0% | 48,0% | ▲▲ fuerte | **Ingreso subió 49%** (175M→260M) mientras costos casi no variaron (combustibles/sueldos estables); apalancamiento operativo favorable. |
| **2235** (FIRST MILE) | 30,0% | 22,0% | ▼ moderada | Ingreso casi igual, pero **mantenimiento vehículos subió** de 31M a 46M. |
| **3310** (LAST MILE COL) | 22,7% | 19,5% | ▼ leve | Ingreso bajó 3% y combustibles subieron (110M→131M) mientras mantenimiento bajó; efecto

## 5 · + WebSearch

In [6]:
cc("¿Qué registra la cuenta 7205 del PUC colombiano? Busca en la web y cita la fuente en una frase.",
   "--system-prompt", SYSTEM, "--disable-slash-commands",
   "--tools", "Bash,Read,Glob,Grep,WebSearch", "--allowedTools", "Bash,Read,Glob,Grep,WebSearch");

$ claude -p '…' --system-prompt 'Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador '"'"';'"'"', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto.' --disable-slash-commands --tools Bash,Read,Glob,Grep,WebSearch --allowedTools Bash,Read,Glob,Grep,WebSearch



La cuenta **7205 – Mano de Obra Directa** (clase 7: Costos de Producción o de Operación) registra los salarios y jornales de los trabajadores vinculados directamente al proceso productivo de bienes.

Fuente: [PUC.com.co - Lista de cuentas de la clase 7](https://puc.com.co/cuentas/clase/7)


## 6 · + skill
`--setting-sources project` carga `.claude/skills/` del cwd. System prompt corto: las reglas están en la skill.

In [7]:
cc(PREGUNTA, "--system-prompt", SYSTEM_CORTO, "--setting-sources", "project",
   "--tools", "Bash,Read,Glob,Grep,Skill", "--allowedTools", "Bash,Read,Glob,Grep,Skill");

$ claude -p '…' --system-prompt 'Eres un analista financiero. Español, corto.' --setting-sources project --tools Bash,Read,Glob,Grep,Skill --allowedTools Bash,Read,Glob,Grep,Skill



**Resumen jun→jul (var. en puntos porcentuales):**

| Proyecto | Junio | Julio | Cambio |
|---|---|---|---|
| 1045 | 33.7% | 11.5% | **-22.2 pp** |
| 2235 | 30.0% | 22.0% | -8.0 pp |
| 3310 | 22.7% | 19.5% | -3.2 pp |
| 4410 | 30.1% | 28.0% | -2.1 pp |
| 3322 | 30.0% | 29.7% | -0.3 pp |
| 1030 | 30.6% | 33.0% | +2.4 pp |
| 2210 | 38.0% | 48.0% | **+10.0 pp** |

**Causas principales:**

- **Proyecto 1045 (mayor caída, -22.2 pp):** disparo en "Costo de Mercancía Vendida" (+$57.5M) y sueldos y salarios (+$19M vs junio). Compensa parcialmente una baja en arriendo bodega y mantenimiento vehículos, pero no alcanza.
- **Proyecto 2235 (-8.0 pp):** mantenimiento de vehículos subió +$14.8M, y no hubo aumento de ingresos que lo compensara.
- **Proyecto 2210 (mejor mejora, +10.0 pp):** margen ya alto y creció más, señal de control de costos o mayor ingreso relativo (revisar detalle si se necesita).

**Nota:** proyectos "0" y "5001" quedan sin margen calculable (sin ingreso registrado en el período

## 7 · + CLAUDE.md (contexto del proyecto)
Mismo comando. Ahora existe `CLAUDE.md` en el cwd y `project` lo carga.

In [8]:
(WS / "CLAUDE.md").write_text("""# Contexto del proyecto
Python: usa `uv run --with pandas python3 -` (pandas no está en el Python del sistema).
Respuesta: tabla por proyecto (junio, julio, cambio en pp; máximo 8 filas), la causa principal en 2 frases, sin introducción ni cierre.
""")
cc(PREGUNTA, "--system-prompt", SYSTEM_CORTO, "--setting-sources", "project",
   "--tools", "Bash,Read,Glob,Grep,Skill", "--allowedTools", "Bash,Read,Glob,Grep,Skill");

$ claude -p '…' --system-prompt 'Eres un analista financiero. Español, corto.' --setting-sources project --tools Bash,Read,Glob,Grep,Skill --allowedTools Bash,Read,Glob,Grep,Skill



La causa principal en el proyecto 1045 (el de mayor caída) es un fuerte aumento en costo de mercancía vendida (+57,5M) y en nómina/sueldos (+19M) de junio a julio. Estos incrementos superaron el crecimiento de ingresos, hundiendo el margen 22 pp.

| Proyecto | Margen jun | Margen jul | Cambio (pp) |
|---|---|---|---|
| 1045 | 33.7% | 11.5% | -22.2 |
| 2235 | 30.0% | 22.0% | -8.0 |
| 3310 | 22.7% | 19.5% | -3.2 |
| 4410 | 30.1% | 28.0% | -2.1 |
| 3322 | 30.0% | 29.7% | -0.3 |
| 1030 | 30.6% | 33.0% | +2.4 |
| 2210 | 38.0% | 48.0% | +10.0 |


## 8 · MVP: salida JSON + límites
`--output-format json` devuelve resultado, costo y turnos: lo que consume un script o una Web UI. `--max-turns` y `--max-budget-usd` acotan.

In [9]:
out = cc(PREGUNTA, "--system-prompt", SYSTEM_CORTO, "--setting-sources", "project",
         "--tools", "Bash,Read,Glob,Grep,Skill", "--allowedTools", "Bash,Read,Glob,Grep,Skill",
         "--output-format", "json", "--max-turns", "20", "--max-budget-usd", "1")
res = json.loads(out)
print({k: res[k] for k in ("num_turns", "total_cost_usd", "duration_ms", "is_error")})

$ claude -p '…' --system-prompt 'Eres un analista financiero. Español, corto.' --setting-sources project --tools Bash,Read,Glob,Grep,Skill --allowedTools Bash,Read,Glob,Grep,Skill --output-format json --max-turns 20 --max-budget-usd 1



{"is_error":false,"duration_api_ms":126235,"num_turns":8,"stop_reason":"end_turn","session_id":"0097f646-a2c5-4e4d-baaf-2942d809445c","total_cost_usd":0.051249,"usage":{"input_tokens":14,"cache_creation_input_tokens":4507,"cache_read_input_tokens":63065,"output_tokens":2058,"output_tokens_details":{"thinking_tokens":161},"server_tool_use":{"web_search_requests":0,"web_fetch_requests":0},"service_tier":"standard","cache_creation":{"ephemeral_1h_input_tokens":4507,"ephemeral_5m_input_tokens":0},"inference_geo":"not_available","iterations":[{"input_tokens":2,"output_tokens":295,"cache_read_input_tokens":10995,"cache_creation_input_tokens":716,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":716},"type":"message"}],"speed":"standard"},"modelUsage":{"claude-sonnet-5":{"inputTokens":14,"outputTokens":2058,"cacheReadInputTokens":63065,"cacheCreationInputTokens":4507,"webSearchRequests":0,"costUSD":0.051249,"contextWindow":1000000,"maxOutputTokens":64000,"canonicalMo

Mismas palancas que el SDK, en flags. `--mcp-config` añade MCP; `--agents` subagentes; `--output-format stream-json` los eventos uno a uno.